# 8.8 Python

Chapter 8 introduces the Beta and Gamma distribution families and develops the theory of order statistics and conjugate priors. This section uses Python to work with those distributions directly, verify the convolution result for sums of Uniforms, simulate the Bayes' billiards story to check the marginal and posterior distributions, and explore order statistics through simulation.

In [ ]:
import numpy as np
import scipy.stats as st
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(palette="deep")
rng = np.random.default_rng(seed=42)

## Beta and Gamma Distributions

`scipy.stats` provides both the Beta and Gamma families. For $X \sim \text{Beta}(a, b)$, the object `st.beta(a, b)` exposes `.pdf`, `.cdf`, and `.rvs`. The Beta distribution is defined on $(0, 1)$, making it the natural home for unknown probabilities.

In [ ]:
a, b = 2, 5
beta_dist = st.beta(a, b)

x_grid = np.linspace(0, 1, 500)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

sns.lineplot(x=x_grid, y=beta_dist.pdf(x_grid), ax=axes[0])
axes[0].set_xlabel("x")
axes[0].set_ylabel("Density")
axes[0].set_title(f"Beta({a}, {b}) PDF")

sns.lineplot(x=x_grid, y=beta_dist.cdf(x_grid), ax=axes[1])
axes[1].set_xlabel("x")
axes[1].set_ylabel("Probability")
axes[1].set_title(f"Beta({a}, {b}) CDF")

plt.tight_layout()
plt.show()

For $X \sim \text{Gamma}(a, \lambda)$, the mean is $a/\lambda$ and the variance is $a/\lambda^2$. `scipy.stats` parameterizes the Gamma by shape `a` and `scale = 1/lambda`, so `st.gamma(a=3, scale=0.5)` represents $\text{Gamma}(3, 2)$. We can verify the theoretical moments by generating a large sample and computing the sample mean and variance.

In [ ]:
a, lam = 3, 2
y = rng.gamma(shape=a, scale=1 / lam, size=10**5)

print(f"Sample mean:     {np.mean(y):.4f}  (true: a/λ = {a/lam})")
print(f"Sample variance: {np.var(y, ddof=1):.4f}  (true: a/λ² = {a/lam**2})")

With $10^5$ draws the sample moments are very close to their true values of $1.5$ and $0.75$.

## Convolution of Uniforms

If $X$ and $Y$ are i.i.d. $\text{Unif}(0, 1)$, the sum $T = X + Y$ has a triangular distribution on $(0, 2)$. This is a special case of the convolution formula for sums of independent random variables. We can verify the triangular shape by simulating a large number of pairs and histogramming their sums.

In [ ]:
x = rng.uniform(size=10**5)
y = rng.uniform(size=10**5)
t = x + y

t_grid = np.linspace(0, 2, 500)
triangular_pdf = np.where(t_grid <= 1, t_grid, 2 - t_grid)

fig, ax = plt.subplots(figsize=(6, 4))
sns.histplot(t, bins=80, stat="density", ax=ax, label="X + Y")
sns.lineplot(x=t_grid, y=triangular_pdf, ax=ax, label="Triangular PDF")
ax.set_xlabel("t")
ax.set_ylabel("Density")
ax.set_title("Sum of two Unif(0, 1) variables")
ax.legend()
plt.tight_layout()
plt.show()

The histogram rises linearly from 0 to 1 and falls symmetrically back to 0, closely tracking the triangular PDF. The agreement improves with more draws.

## Bayes' Billiards

The Bayes' billiards story describes a two-stage experiment: a gray ball lands uniformly at random on $[0, 1]$, establishing an unknown position $p$; then $n$ white balls are thrown independently, each landing to the left of the gray ball with probability $p$. Letting $X$ count the white balls to the left,

$$p \sim \text{Unif}(0, 1), \qquad X \mid p \sim \text{Bin}(n, p).$$

Two facts follow from this chapter's theory:
1. The marginal distribution of $X$ is $\text{DiscreteUnif}\{0, 1, \ldots, n\}$.
2. The posterior $p \mid X = x$ is $\text{Beta}(x + 1,\; n - x + 1)$.

We simulate $10^5$ repetitions of the experiment to check both facts. The key is that `rng.binomial` accepts a vector of probabilities, drawing each $X_i$ from $\text{Bin}(n, p_i)$ independently.

In [ ]:
nsim = 10**5
n = 10

p_sim = rng.uniform(size=nsim)
x_sim = rng.binomial(n=n, p=p_sim)

# Marginal distribution of X: expect Discrete Uniform on {0, 1, ..., 10}
fig, ax = plt.subplots(figsize=(7, 4))
sns.histplot(x_sim, bins=np.arange(-0.5, n + 1.5), stat="probability", ax=ax)
ax.axhline(1 / (n + 1), color="black", linestyle="--", label=f"1/(n+1) = {1/(n+1):.3f}")
ax.set_xlabel("x")
ax.set_ylabel("Proportion")
ax.set_title("Marginal distribution of X (should be Discrete Uniform)")
ax.legend()
plt.tight_layout()
plt.show()

Each bar is near $1/11 \approx 0.091$, the uniform probability, confirming that the marginal of $X$ is discrete uniform on $\{0, 1, \ldots, 10\}$.

Now for the posterior. To isolate the simulated values of $p$ that correspond to $X = 3$, we use boolean indexing: `p_sim[x_sim == 3]`. According to the Beta-Binomial conjugacy result, this conditional sample should look like draws from $\text{Beta}(4, 8)$.

In [ ]:
x_obs = 3
p_posterior = p_sim[x_sim == x_obs]   # simulated p values where X = 3

# Theoretical posterior: Beta(x+1, n-x+1) = Beta(4, 8)
post_dist = st.beta(x_obs + 1, n - x_obs + 1)
p_grid = np.linspace(0, 1, 500)

fig, ax = plt.subplots(figsize=(6, 4))
sns.histplot(p_posterior, bins=40, stat="density", ax=ax, label="Simulated posterior")
sns.lineplot(x=p_grid, y=post_dist.pdf(p_grid), ax=ax, label="Beta(4, 8) PDF")
ax.set_xlabel("p")
ax.set_ylabel("Density")
ax.set_title(r"Posterior $p \mid X = 3$ vs. Beta(4, 8)")
ax.legend()
plt.tight_layout()
plt.show()

The simulated posterior histogram aligns closely with the $\text{Beta}(4, 8)$ PDF, validating the conjugacy result. The posterior is concentrated below $0.5$ because observing only 3 successes in 10 trials is evidence that $p$ is small.

## Order Statistics

The $k$th order statistic $X_{(k)}$ of a sample $X_1, \ldots, X_n$ is the $k$th smallest value. Simulating order statistics requires nothing beyond sorting: generate an i.i.d. sample and sort it. One realization of all ten order statistics from a $N(0,1)$ sample of size 10:

In [ ]:
one_sample = np.sort(rng.normal(size=10))
for k, val in enumerate(one_sample, start=1):
    print(f"X({k:2d}) = {val:+.4f}")

To study the distribution of a particular order statistic, we repeat the sorting experiment many times. Each call to `np.sort(rng.normal(size=10))` yields a full sorted sample; stacking $10^4$ of them gives a matrix whose columns are the $10^4$ realizations of each $X_{(k)}$.

In [ ]:
order_stats = np.array([np.sort(rng.normal(size=10)) for _ in range(10**4)])
# order_stats is (10000, 10): row i is one sorted sample, column k is X_{(k+1)}

x9 = order_stats[:, 8]   # 0-based index 8 → X_{(9)}, the 9th smallest

print(f"X(9) sample mean:     {np.mean(x9):.4f}")
print(f"X(9) sample variance: {np.var(x9, ddof=1):.4f}")

The 9th order statistic of 10 i.i.d. standard Normals tends to be positive — it is the second-largest value and is typically near the upper tail. We can plot its distribution and overlay the theoretical PDF, which for the $k$th order statistic of i.i.d. draws from a distribution with PDF $f$ and CDF $F$ is

$$f_{X_{(k)}}(x) = \frac{n!}{(k-1)!(n-k)!}\, [F(x)]^{k-1}\, [1-F(x)]^{n-k}\, f(x).$$

In [ ]:
from math import factorial

k, n_os = 9, 10
coeff = factorial(n_os) / (factorial(k - 1) * factorial(n_os - k))

x_grid = np.linspace(-1, 4, 500)
pdf_x9 = coeff * st.norm.cdf(x_grid)**(k-1) * (1 - st.norm.cdf(x_grid))**(n_os-k) * st.norm.pdf(x_grid)

fig, ax = plt.subplots(figsize=(6, 4))
sns.histplot(x9, bins=60, stat="density", ax=ax, label=r"Simulated $X_{(9)}$")
sns.lineplot(x=x_grid, y=pdf_x9, ax=ax, label=r"Theoretical PDF of $X_{(9)}$")
ax.set_xlabel("x")
ax.set_ylabel("Density")
ax.set_title(r"Distribution of $X_{(9)}$ from $N(0,1)$ samples of size 10")
ax.legend()
plt.tight_layout()
plt.show()